## Configuración del entorno

In [2]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split

# Agregar ruta de proyecto al path
PROJECT_ROOT = Path('..').resolve()
NOTEBOOK_DIR = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'Bloque2_CNN'))

# Importar desde nuestro data_loader centralizado
from src.data_loader import DMIDMultiTaskDataset, create_data_loaders

# Parámetros globales - RUTAS CORRECTAS
IMG_SIZE = (256, 256)  # Según config.yaml
DMID_PNG_DIR = PROJECT_ROOT / 'DMID_PNG'
METADATA_PATH = PROJECT_ROOT / 'Metadata.xlsx'

print(f"✓ Rutas validadas:")
print(f"  - Project Root: {PROJECT_ROOT}")
print(f"  - DMID_PNG: {DMID_PNG_DIR} (exists: {DMID_PNG_DIR.exists()})")
print(f"  - Metadata: {METADATA_PATH} (exists: {METADATA_PATH.exists()})")


ModuleNotFoundError: No module named 'torch'

## Pipeline de Augmentation

In [ ]:
"""
Pipeline de Augmentation optimizado para imágenes mamarias.
Mantiene transformaciones realistas y documentadas.
"""

train_transform = A.Compose([
    # Resize a tamaño estándar
    A.Resize(height=IMG_SIZE[0], width=IMG_SIZE[1]),
    
    # Data Augmentation para tejidos biológicos
    A.Rotate(limit=15, p=0.5),                    # Rotación ±15°
    A.HorizontalFlip(p=0.5),                      # Flip horizontal
    A.VerticalFlip(p=0.2),                        # Flip vertical (conservador)
    
    # Deformaciones elásticas - MUY ÚTIL en tejidos biológicos
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    
    # Perturbaciones sutiles
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.RandomScale(scale_limit=0.1, p=0.3),
    
    # Normalización a [0, 1] - escala 8-bit a float
    A.Normalize(mean=(0.0,), std=(1.0,), max_pixel_value=255.0),
    ToTensorV2()
])

# Para validación y test: solo resize + normalización (sin augmentation)
val_test_transform = A.Compose([
    A.Resize(height=IMG_SIZE[0], width=IMG_SIZE[1]),
    A.Normalize(mean=(0.0,), std=(1.0,), max_pixel_value=255.0),
    ToTensorV2()
])

print("✓ Pipelines de transformación creados")


## Carga del Dataset y Verificación de Rango

In [ ]:
# Cargar datos usando nuestro data_loader centralizado
# Esto garantiza consistencia con train.py y evaluate.py

# Crear dataset multi-task (soporta clasificación, detección, segmentación)
dataset = DMIDMultiTaskDataset(
    image_dir=str(DMID_PNG_DIR),
    metadata_path=str(METADATA_PATH),
    split='all',  # Cargar todo para exploración
    transform=train_transform,
    return_dict=True
)

print(f"✓ Dataset cargado:")
print(f"  - Total imágenes: {len(dataset)}")
print(f"  - Clase distribution: {dataset.class_distribution if hasattr(dataset, 'class_distribution') else 'N/A'}")

# Verificar un sample
if len(dataset) > 0:
    sample = dataset[0]
    sample_img = sample['image']
    
    print(f"\n✓ Estadísticas del sample:")
    print(f"  - Tipo: {type(sample_img)}")
    print(f"  - Shape: {sample_img.shape}")
    print(f"  - Rango: [{sample_img.min():.4f}, {sample_img.max():.4f}]")
    print(f"  - dtype: {sample_img.dtype}")
    
    if 'mask' in sample and sample['mask'] is not None:
        mask = sample['mask']
        print(f"  - Mask shape: {mask.shape}")
        print(f"  - Mask unique: {torch.unique(mask)}")


## Visualización de Ejemplos Augmentados

In [ ]:
"""
Visualización de ejemplos augmentados para validar el pipeline.
Muestra imagen original + máscara (cuando esté disponible).
"""

def visualize_augmentations(dataset, samples=5):
    """
    Visualiza samples aleatorios del dataset con sus transformaciones.
    
    Args:
        dataset: DMIDMultiTaskDataset instance
        samples: número de ejemplos a mostrar
    """
    fig, axes = plt.subplots(samples, 3, figsize=(15, 4*samples))
    if samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(samples):
        idx = np.random.randint(len(dataset))
        sample = dataset[idx]
        
        image = sample['image']
        image_np = image.squeeze().numpy() if torch.is_tensor(image) else image
        
        # Fila 1: Imagen
        axes[i, 0].imshow(image_np, cmap='gray')
        axes[i, 0].set_title(f"Sample {i+1} - Imagen")
        axes[i, 0].axis('off')
        
        # Fila 2: Máscara (si existe)
        if 'mask' in sample and sample['mask'] is not None:
            mask = sample['mask']
            mask_np = mask.squeeze().numpy() if torch.is_tensor(mask) else mask
            axes[i, 1].imshow(mask_np, cmap='jet')
            axes[i, 1].set_title(f"Sample {i+1} - Máscara")
            axes[i, 1].axis('off')
            
            # Fila 3: Overlay imagen + máscara
            axes[i, 2].imshow(image_np, cmap='gray', alpha=0.7)
            axes[i, 2].imshow(mask_np, cmap='jet', alpha=0.3)
            axes[i, 2].set_title(f"Sample {i+1} - Overlay")
            axes[i, 2].axis('off')
        else:
            axes[i, 1].text(0.5, 0.5, 'Sin máscara', ha='center', va='center')
            axes[i, 1].axis('off')
            axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Ejecutar visualización
print(f"Visualizando {min(5, len(dataset))} augmentaciones aleatorias...")
visualize_augmentations(dataset, samples=min(5, len(dataset)))


NameError: name 'dataset' is not defined

## División del Dataset (60/20/20)

In [ ]:
"""
Crear DataLoaders usando el framework centralizado.
Esto garantiza consistencia entre notebooks y train.py/evaluate.py.
"""

# Usar la función centralizada que ya hace stratified k-fold
loaders = create_data_loaders(
    config_path=str(PROJECT_ROOT / 'Bloque2_CNN' / 'config.yaml'),
    batch_size=16,
    num_workers=4,
    shuffle_train=True
)

print("✓ DataLoaders creados desde config.yaml")
print(f"  - Train batches: {len(loaders['train'])}")
print(f"  - Val batches: {len(loaders['val'])}")
print(f"  - Test batches: {len(loaders['test']) if 'test' in loaders else 'N/A'}")

# Verificar un batch
train_batch = next(iter(loaders['train']))
print(f"\n✓ Sample batch:")
print(f"  - Keys: {train_batch.keys()}")
print(f"  - Image shape: {train_batch['image'].shape}")
print(f"  - Label shape: {train_batch['label'].shape if 'label' in train_batch else 'N/A'}")
print(f"  - Image range: [{train_batch['image'].min():.4f}, {train_batch['image'].max():.4f}]")

# Información de distribución
if 'class_distribution' in train_batch:
    print(f"\n✓ Class distribution (train):")
    for cls, count in train_batch['class_distribution'].items():
        print(f"  - {cls}: {count}")


In [ ]:
"""
Demostración: Entrenar clasificación con estos DataLoaders
"""

# Importar trainer
from src.train import ClassificationTrainer

# Crear trainer con nuestros dataloaders
trainer = ClassificationTrainer(config=str(PROJECT_ROOT / 'Bloque2_CNN' / 'config.yaml'))

print("✓ Trainer creado. Para entrenar:")
print("  trainer.fit(fold=0)  # Entrena fold 0 de 5")
print("\nO desde terminal:")
print("  python quick_start.py train --task classification")

# Notas:
print("\n📝 NOTAS:")
print("  - Carga automáticamente DMID_PNG desde config.yaml")
print("  - Usa k-fold stratified cross-validation")
print("  - Early stopping con patience=20 epochs")
print("  - Checkpoints guardados en models/classification/")
print("  - Mejor modelo guardado como best.pth")


## Integración con train.py

Este notebook demuestra el pipeline de preprocesamiento. Para entrenar modelos, usa:

```bash
# Opción 1: Desde terminal
python quick_start.py train --task all

# Opción 2: Importar directamente
from src.train import ClassificationTrainer, DetectionTrainer, SegmentationTrainer
trainer = ClassificationTrainer(config='config.yaml')
trainer.fit(fold=0)
```

Ver [TRAINING_GUIDE.md](../TRAINING_GUIDE.md) para más detalles.
